# Cleaned-Statement Cluster Sensitivity (Supplementary Table S4)

Self-contained: all analysis code is defined in this notebook (no external module).

Cluster-robust standard errors and cluster bootstrap intervals keyed by the cleaned statement field, matching the grouped-split normalization used by the modeling notebooks.

In [1]:
from pathlib import Path
import json

try:
    from google.colab import drive
    drive.mount('/content/drive')
    R2_ROOT = Path('.')
    ANALYSIS_DIR = R2_ROOT / '07_ValidationRobustness/05_cluster_sensitivity'
except ImportError:
    ANALYSIS_DIR = Path.cwd().resolve()
    R2_ROOT = ANALYSIS_DIR.parents[1]

OUTPUT = ANALYSIS_DIR / 'outputs'
print(f'R2 root: {R2_ROOT}')
print(f'Outputs: {OUTPUT}')

Mounted at /content/drive
R2 root: .
Outputs: ./07_ValidationRobustness/05_cluster_sensitivity/outputs


In [2]:
#!/usr/bin/env python3
"""Cluster-aware sensitivity analyses for the MentalHealth R2 package.

The cleaned statement is the resampling/covariance cluster throughout.  These
analyses condition on the already fitted seed-2025 prediction files; they
quantify held-out sampling variation under duplicate-text clustering and do
not estimate training-run variability.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import platform
import sys
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import scipy
from scipy.stats import chi2_contingency
import sklearn
from sklearn.metrics import roc_auc_score
import statsmodels
import statsmodels.formula.api as smf
import re



# --- canonical entropy/cleaning implementation, inlined (formerly a shared module) ---

NOTEBOOK_NAME = "MentalHealth_R2_cluster_sensitivity.ipynb"

N_CLASSES = 7


PROB_COLS = (
    "u_p_normal",
    "u_p_depression",
    "u_p_anxiety",
    "u_p_suicidal",
    "u_p_stress",
    "u_p_bipolar",
    "u_p_personality_disorder",
)


def clean_text(text: object) -> str:
    """Match the grouped-split statement normalization used in R2."""
    if pd.isna(text):
        return ""
    value = str(text).lower()
    value = re.sub(r"http\S+", " urltoken ", value)
    value = re.sub(r"@\w+", " usertoken ", value)
    value = re.sub(r"#(\w+)", r" hashtag_\1 ", value)
    value = re.sub(r"[^\w\s]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def normalize_label(series: pd.Series) -> pd.Series:
    """Normalize released and protocol hard-label spellings."""
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"[\s-]+", "_", regex=True)
    )


def normalize_score_rows(scores: np.ndarray) -> np.ndarray:
    """Validate and row-normalize nonnegative class-score vectors."""
    values = np.asarray(scores, dtype=float)
    if values.ndim != 2:
        raise ValueError(f"Score matrix must be two-dimensional; got {values.ndim}D")
    if values.shape[1] != N_CLASSES:
        raise ValueError(
            f"Expected {N_CLASSES} score columns; found {values.shape[1]}"
        )
    if not np.isfinite(values).all():
        raise ValueError("Score vectors contain non-finite values")
    if (values < 0).any():
        raise ValueError("Score vectors contain negative values")
    row_sums = values.sum(axis=1)
    if (row_sums <= 0).any():
        bad_n = int((row_sums <= 0).sum())
        raise ValueError(f"Found {bad_n} score vectors with non-positive row sums")
    normalized = values / row_sums[:, None]
    if not np.allclose(normalized.sum(axis=1), 1.0, rtol=0.0, atol=1e-12):
        raise AssertionError("Row-normalized score vectors do not sum to one")
    return normalized


def shannon_entropy(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return normalized rows, entropy in nats, and normalized entropy.

    Zeros are not clipped.  Terms with probability zero are assigned exactly
    zero, implementing the limiting convention ``0 * log(0) = 0``.
    """
    probs = normalize_score_rows(scores)
    terms = np.zeros_like(probs)
    positive = probs > 0
    terms[positive] = -probs[positive] * np.log(probs[positive])
    entropy_nats = terms.sum(axis=1)
    entropy_normalized = entropy_nats / math.log(probs.shape[1])
    if (entropy_nats < 0).any() or (entropy_nats > math.log(N_CLASSES) + 1e-12).any():
        raise AssertionError("Entropy fell outside its mathematical bounds")
    return probs, entropy_nats, entropy_normalized


class _CanonicalEntropy:
    """Namespace shim so the analysis code below keeps its original call sites."""
    PROB_COLS = PROB_COLS
    clean_text = staticmethod(clean_text)
    normalize_label = staticmethod(normalize_label)
    shannon_entropy = staticmethod(shannon_entropy)
    __file__ = "MentalHealth_R2_cluster_sensitivity.ipynb"

EXPECTED_N = 53_043
F1_BOOTSTRAP_RESAMPLES = 1_000
OR_BOOTSTRAP_RESAMPLES = 2_000
BOOTSTRAP_SEED = 2025

EXPECTED_LABELS = (
    "ANXIETY",
    "BIPOLAR",
    "DEPRESSION",
    "NORMAL",
    "PERSONALITY_DISORDER",
    "STRESS",
    "SUICIDAL",
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_canonical_entropy_module(r2_root: Path):
    """Return the inlined canonical implementation (no external module)."""
    return _CanonicalEntropy()


def f1_from_confusion(confusion: np.ndarray) -> tuple[float, float]:
    """Return weighted and macro F1 from a square confusion matrix."""
    matrix = np.asarray(confusion, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("Confusion matrix must be square")
    if not np.isfinite(matrix).all() or (matrix < 0).any():
        raise ValueError("Confusion matrix must be finite and nonnegative")
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    denominator = support + predicted
    f1 = np.divide(
        2.0 * np.diag(matrix),
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 0,
    )
    total = support.sum()
    if total <= 0:
        raise ValueError("Confusion matrix has no observations")
    weighted = float(np.sum(f1 * support) / total)
    macro = float(np.mean(f1))
    return weighted, macro


def f1_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int,
    row_weights: np.ndarray | None = None,
) -> tuple[float, float]:
    """Compute fixed-label weighted and macro F1, optionally with row weights."""
    truth = np.asarray(y_true, dtype=int)
    pred = np.asarray(y_pred, dtype=int)
    if truth.ndim != 1 or pred.ndim != 1 or len(truth) != len(pred):
        raise ValueError("y_true and y_pred must be equal-length vectors")
    if ((truth < 0) | (truth >= n_classes)).any():
        raise ValueError("y_true contains an out-of-range class")
    if ((pred < 0) | (pred >= n_classes)).any():
        raise ValueError("y_pred contains an out-of-range class")
    encoded = truth * n_classes + pred
    weights = None if row_weights is None else np.asarray(row_weights, dtype=float)
    if weights is not None and (len(weights) != len(truth) or (weights < 0).any()):
        raise ValueError("row_weights must be nonnegative and match y_true")
    confusion = np.bincount(
        encoded,
        weights=weights,
        minlength=n_classes * n_classes,
    ).reshape(n_classes, n_classes)
    return f1_from_confusion(confusion)


def factorize_groups(groups: pd.Series) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Return row-to-group codes, cluster sizes, and cluster diagnostics."""
    if groups.isna().any():
        raise ValueError("Cleaned-statement groups contain missing values")
    codes, unique = pd.factorize(groups, sort=False)
    if (codes < 0).any():
        raise AssertionError("Group factorization produced missing codes")
    sizes = np.bincount(codes, minlength=len(unique))
    diagnostics = pd.DataFrame(
        [
            {
                "n_rows": int(len(groups)),
                "n_cleaned_statement_clusters": int(len(unique)),
                "n_duplicate_clusters": int((sizes > 1).sum()),
                "n_rows_in_duplicate_clusters": int(sizes[sizes > 1].sum()),
                "duplicate_row_fraction": float(sizes[sizes > 1].sum() / len(groups)),
                "maximum_cluster_size": int(sizes.max()),
            }
        ]
    )
    return codes, sizes, diagnostics


def label_codes(values: pd.Series, labels: tuple[str, ...], normalize_label) -> np.ndarray:
    """Normalize string labels and encode them against a fixed label set."""
    normalized = normalize_label(values)
    mapping = {label: index for index, label in enumerate(labels)}
    unexpected = set(normalized.unique()) - set(labels)
    if unexpected:
        raise ValueError(f"Unexpected labels: {sorted(unexpected)}")
    return normalized.map(mapping).to_numpy(dtype=int)


def bootstrap_common_target_f1(
    *,
    analysis: str,
    panel: str,
    groups: pd.Series,
    y_true: np.ndarray,
    predictions: dict[str, np.ndarray],
    n_classes: int,
    resamples: int = F1_BOOTSTRAP_RESAMPLES,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Cluster-bootstrap F1 for models sharing a target and test set."""
    group_codes, _, group_diagnostics = factorize_groups(groups)
    n_groups = int(group_codes.max() + 1)
    rng = np.random.default_rng(seed)
    point = {
        model: f1_metrics(y_true, pred, n_classes)
        for model, pred in predictions.items()
    }
    replicate_rows: list[dict[str, object]] = []
    for replicate in range(resamples):
        sampled_groups = rng.integers(0, n_groups, size=n_groups)
        cluster_multiplicity = np.bincount(sampled_groups, minlength=n_groups)
        row_weights = cluster_multiplicity[group_codes]
        bootstrap_n = int(row_weights.sum())
        for model, pred in predictions.items():
            weighted, macro = f1_metrics(y_true, pred, n_classes, row_weights)
            replicate_rows.append(
                {
                    "analysis": analysis,
                    "panel": panel,
                    "replicate": replicate + 1,
                    "model": model,
                    "bootstrap_n_rows": bootstrap_n,
                    "weighted_f1": weighted,
                    "macro_f1": macro,
                }
            )
    replicates = pd.DataFrame(replicate_rows)
    summary_rows: list[dict[str, object]] = []
    for model, (point_weighted, point_macro) in point.items():
        part = replicates.loc[replicates["model"] == model]
        for metric, point_value in (
            ("weighted_f1", point_weighted),
            ("macro_f1", point_macro),
        ):
            summary_rows.append(
                {
                    "analysis": analysis,
                    "panel": panel,
                    "level": "model",
                    "model": model,
                    "aspect": "",
                    "metric": metric,
                    "point_estimate": point_value,
                    "cluster_bootstrap_ci_95_lower": float(part[metric].quantile(0.025)),
                    "cluster_bootstrap_ci_95_upper": float(part[metric].quantile(0.975)),
                    "test_n_rows": int(len(y_true)),
                    "test_n_cleaned_statement_clusters": n_groups,
                    "bootstrap_resamples": resamples,
                    "bootstrap_seed": seed,
                    "inference_scope": "fixed fitted model; cleaned-statement cluster resampling",
                }
            )
    diagnostics = group_diagnostics.assign(analysis=analysis, panel=panel)
    return pd.DataFrame(summary_rows), replicates, diagnostics


def bootstrap_aspect_f1(
    *,
    groups: pd.Series,
    tasks: dict[tuple[str, str], tuple[np.ndarray, np.ndarray]],
    resamples: int = F1_BOOTSTRAP_RESAMPLES,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Cluster-bootstrap per-aspect and shared-replicate mean F1."""
    group_codes, _, group_diagnostics = factorize_groups(groups)
    n_groups = int(group_codes.max() + 1)
    rng = np.random.default_rng(seed)
    point = {
        key: f1_metrics(y_true, y_pred, 3)
        for key, (y_true, y_pred) in tasks.items()
    }
    models = sorted({model for model, _ in tasks})
    aspects = sorted({aspect for _, aspect in tasks})
    replicate_rows: list[dict[str, object]] = []
    for replicate in range(resamples):
        sampled_groups = rng.integers(0, n_groups, size=n_groups)
        multiplicity = np.bincount(sampled_groups, minlength=n_groups)
        row_weights = multiplicity[group_codes]
        bootstrap_n = int(row_weights.sum())
        replicate_metrics: dict[tuple[str, str], tuple[float, float]] = {}
        for (model, aspect), (y_true, y_pred) in tasks.items():
            weighted, macro = f1_metrics(y_true, y_pred, 3, row_weights)
            replicate_metrics[(model, aspect)] = (weighted, macro)
            replicate_rows.append(
                {
                    "analysis": "Table IV",
                    "panel": "aspect_heads",
                    "replicate": replicate + 1,
                    "level": "aspect",
                    "model": model,
                    "aspect": aspect,
                    "bootstrap_n_rows": bootstrap_n,
                    "weighted_f1": weighted,
                    "macro_f1": macro,
                }
            )
        for model in models:
            mean_weighted = float(
                np.mean([replicate_metrics[(model, aspect)][0] for aspect in aspects])
            )
            mean_macro = float(
                np.mean([replicate_metrics[(model, aspect)][1] for aspect in aspects])
            )
            replicate_rows.append(
                {
                    "analysis": "Table IV",
                    "panel": "aspect_heads",
                    "replicate": replicate + 1,
                    "level": "mean_across_six_aspects",
                    "model": model,
                    "aspect": "MEAN",
                    "bootstrap_n_rows": bootstrap_n,
                    "weighted_f1": mean_weighted,
                    "macro_f1": mean_macro,
                }
            )
    replicates = pd.DataFrame(replicate_rows)

    summary_rows: list[dict[str, object]] = []
    for (model, aspect), (point_weighted, point_macro) in point.items():
        part = replicates.loc[
            (replicates["level"] == "aspect")
            & (replicates["model"] == model)
            & (replicates["aspect"] == aspect)
        ]
        for metric, point_value in (
            ("weighted_f1", point_weighted),
            ("macro_f1", point_macro),
        ):
            summary_rows.append(
                {
                    "analysis": "Table IV",
                    "panel": "aspect_heads",
                    "level": "aspect",
                    "model": model,
                    "aspect": aspect,
                    "metric": metric,
                    "point_estimate": point_value,
                    "cluster_bootstrap_ci_95_lower": float(part[metric].quantile(0.025)),
                    "cluster_bootstrap_ci_95_upper": float(part[metric].quantile(0.975)),
                    "test_n_rows": len(groups),
                    "test_n_cleaned_statement_clusters": n_groups,
                    "bootstrap_resamples": resamples,
                    "bootstrap_seed": seed,
                    "inference_scope": "fixed fitted model; cleaned-statement cluster resampling",
                }
            )
    for model in models:
        part = replicates.loc[
            (replicates["level"] == "mean_across_six_aspects")
            & (replicates["model"] == model)
        ]
        point_weighted = float(np.mean([point[(model, aspect)][0] for aspect in aspects]))
        point_macro = float(np.mean([point[(model, aspect)][1] for aspect in aspects]))
        for metric, point_value in (
            ("weighted_f1", point_weighted),
            ("macro_f1", point_macro),
        ):
            summary_rows.append(
                {
                    "analysis": "Table IV",
                    "panel": "aspect_heads",
                    "level": "mean_across_six_aspects",
                    "model": model,
                    "aspect": "MEAN",
                    "metric": metric,
                    "point_estimate": point_value,
                    "cluster_bootstrap_ci_95_lower": float(part[metric].quantile(0.025)),
                    "cluster_bootstrap_ci_95_upper": float(part[metric].quantile(0.975)),
                    "test_n_rows": len(groups),
                    "test_n_cleaned_statement_clusters": n_groups,
                    "bootstrap_resamples": resamples,
                    "bootstrap_seed": seed,
                    "inference_scope": "fixed fitted model; cleaned-statement cluster resampling",
                }
            )
    diagnostics = group_diagnostics.assign(analysis="Table IV", panel="aspect_heads")
    return pd.DataFrame(summary_rows), replicates, diagnostics


def fit_table_i_cluster_robust(
    data_path: Path,
    canonical,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Refit Table I with cleaned-statement cluster-robust covariance."""
    columns = ["statement", "status", "u_label", *canonical.PROB_COLS]
    df = pd.read_csv(data_path, usecols=columns, low_memory=False).reset_index(drop=True)
    if len(df) != EXPECTED_N:
        raise ValueError(f"Expected {EXPECTED_N:,} rows; found {len(df):,}")
    _, entropy_nats, _ = canonical.shannon_entropy(
        df.loc[:, canonical.PROB_COLS].to_numpy(dtype=float)
    )
    df["released_label"] = canonical.normalize_label(df["status"])
    df["ai_hard_label"] = canonical.normalize_label(df["u_label"])
    df["disagreement"] = (df["released_label"] != df["ai_hard_label"]).astype(int)
    df["clean_statement"] = df["statement"].map(canonical.clean_text)
    df["entropy_nats"] = entropy_nats
    entropy_sd = float(df["entropy_nats"].std(ddof=1))
    df["entropy_std"] = (
        df["entropy_nats"] - df["entropy_nats"].mean()
    ) / entropy_sd
    group_codes, cluster_sizes, diagnostics = factorize_groups(df["clean_statement"])

    specs = {
        "entropy_only": "disagreement ~ entropy_std",
        "entropy_plus_released_status": (
            "disagreement ~ entropy_std + "
            "C(released_label, Treatment(reference='NORMAL'))"
        ),
    }
    coefficient_rows: list[dict[str, object]] = []
    model_rows: list[dict[str, object]] = []
    y = df["disagreement"].to_numpy(dtype=int)
    for model_name, formula in specs.items():
        model_based = smf.logit(formula, data=df).fit(disp=0)
        cluster_robust = smf.logit(formula, data=df).fit(
            disp=0,
            cov_type="cluster",
            cov_kwds={
                "groups": group_codes,
                "use_correction": True,
                "df_correction": True,
            },
        )
        if not np.allclose(model_based.params, cluster_robust.params, rtol=0, atol=1e-12):
            raise AssertionError("Cluster covariance changed Table I point estimates")
        robust_ci = cluster_robust.conf_int(alpha=0.05)
        model_ci = model_based.conf_int(alpha=0.05)
        for term in model_based.params.index:
            coefficient_rows.append(
                {
                    "model": model_name,
                    "term": term,
                    "coefficient_log_odds": float(model_based.params[term]),
                    "odds_ratio": float(np.exp(model_based.params[term])),
                    "model_based_se": float(model_based.bse[term]),
                    "model_based_p_value": float(model_based.pvalues[term]),
                    "model_based_or_ci_95_lower": float(np.exp(model_ci.loc[term, 0])),
                    "model_based_or_ci_95_upper": float(np.exp(model_ci.loc[term, 1])),
                    "cluster_robust_se": float(cluster_robust.bse[term]),
                    "cluster_robust_p_value": float(cluster_robust.pvalues[term]),
                    "cluster_robust_or_ci_95_lower": float(
                        np.exp(robust_ci.loc[term, 0])
                    ),
                    "cluster_robust_or_ci_95_upper": float(
                        np.exp(robust_ci.loc[term, 1])
                    ),
                }
            )
        if model_name == "entropy_only":
            auc = float(roc_auc_score(y, df["entropy_nats"]))
            auc_basis = "raw_entropy_predictor_nats"
        else:
            auc = float(roc_auc_score(y, model_based.predict(df)))
            auc_basis = "fitted_model_probability"
        model_rows.append(
            {
                "model": model_name,
                "formula": formula,
                "n_rows": int(model_based.nobs),
                "n_cleaned_statement_clusters": int(len(cluster_sizes)),
                "maximum_cluster_size": int(cluster_sizes.max()),
                "covariance": "cleaned-statement cluster-robust sandwich",
                "small_sample_correction": True,
                "mcfadden_pseudo_r2": float(model_based.prsquared),
                "auc_tie_aware": auc,
                "auc_score_basis": auc_basis,
                "entropy_mean_nats": float(df["entropy_nats"].mean()),
                "entropy_sd_nats_ddof1": entropy_sd,
            }
        )
    diagnostics = diagnostics.assign(analysis="Table I", panel="full_corpus")
    return pd.DataFrame(coefficient_rows), pd.DataFrame(model_rows), diagnostics


def cooccurrence_cluster_bootstrap(
    data_path: Path,
    canonical,
    resamples: int = OR_BOOTSTRAP_RESAMPLES,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Cluster-bootstrap the full-corpus co-occurrence/disagreement odds ratio."""
    columns = ["statement", "status", "u_label", "u_n_conditions"]
    df = pd.read_csv(data_path, usecols=columns, low_memory=False).reset_index(drop=True)
    if len(df) != EXPECTED_N:
        raise ValueError(f"Expected {EXPECTED_N:,} rows; found {len(df):,}")
    df["released_label"] = canonical.normalize_label(df["status"])
    df["ai_hard_label"] = canonical.normalize_label(df["u_label"])
    df["disagreement"] = (df["released_label"] != df["ai_hard_label"]).astype(int)
    conditions = pd.to_numeric(df["u_n_conditions"], errors="raise")
    df["cooccurrence"] = (conditions >= 2).astype(int)
    df["clean_statement"] = df["statement"].map(canonical.clean_text)
    group_codes, _, diagnostics = factorize_groups(df["clean_statement"])
    n_groups = int(group_codes.max() + 1)

    # Cell order: cooccur+disagree, cooccur+agree, noncooccur+disagree,
    # noncooccur+agree.
    cell = np.select(
        [
            (df["cooccurrence"] == 1) & (df["disagreement"] == 1),
            (df["cooccurrence"] == 1) & (df["disagreement"] == 0),
            (df["cooccurrence"] == 0) & (df["disagreement"] == 1),
        ],
        [0, 1, 2],
        default=3,
    ).astype(int)
    group_cell_counts = np.zeros((n_groups, 4), dtype=np.int64)
    np.add.at(group_cell_counts, (group_codes, cell), 1)
    total_counts = group_cell_counts.sum(axis=0)

    def odds_ratio(counts: np.ndarray) -> float:
        a, b, c, d = counts.astype(float)
        if min(a, b, c, d) <= 0:
            return math.nan
        return float((a * d) / (b * c))

    point_or = odds_ratio(total_counts)
    a, b, c, d = total_counts.astype(float)
    log_se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    row_ci = (math.exp(math.log(point_or) - 1.96 * log_se), math.exp(math.log(point_or) + 1.96 * log_se))
    contingency = np.array([[d, c], [b, a]], dtype=int)
    chi2, chi2_p, _, _ = chi2_contingency(contingency, correction=True)

    rng = np.random.default_rng(seed)
    replicate_rows: list[dict[str, object]] = []
    for replicate in range(resamples):
        sampled = rng.integers(0, n_groups, size=n_groups)
        counts = group_cell_counts[sampled].sum(axis=0)
        estimate = odds_ratio(counts)
        if not np.isfinite(estimate):
            raise AssertionError("A co-occurrence bootstrap replicate had an empty cell")
        replicate_rows.append(
            {
                "replicate": replicate + 1,
                "cooccurrence_disagreement_n": int(counts[0]),
                "cooccurrence_agreement_n": int(counts[1]),
                "noncooccurrence_disagreement_n": int(counts[2]),
                "noncooccurrence_agreement_n": int(counts[3]),
                "bootstrap_n_rows": int(counts.sum()),
                "odds_ratio": estimate,
            }
        )
    replicates = pd.DataFrame(replicate_rows)
    summary = pd.DataFrame(
        [
            {
                "analysis": "cooccurrence_vs_forced_choice_disagreement",
                "definition": "u_n_conditions >= 2 versus 0 or 1",
                "n_rows": int(len(df)),
                "n_cleaned_statement_clusters": n_groups,
                "cooccurrence_disagreement_n": int(total_counts[0]),
                "cooccurrence_agreement_n": int(total_counts[1]),
                "noncooccurrence_disagreement_n": int(total_counts[2]),
                "noncooccurrence_agreement_n": int(total_counts[3]),
                "cooccurrence_disagreement_rate": float(a / (a + b)),
                "noncooccurrence_disagreement_rate": float(c / (c + d)),
                "row_level_odds_ratio": point_or,
                "row_level_wald_ci_95_lower": row_ci[0],
                "row_level_wald_ci_95_upper": row_ci[1],
                "yates_chi_square": float(chi2),
                "yates_p_value": float(chi2_p),
                "cluster_bootstrap_ci_95_lower": float(
                    replicates["odds_ratio"].quantile(0.025)
                ),
                "cluster_bootstrap_ci_95_upper": float(
                    replicates["odds_ratio"].quantile(0.975)
                ),
                "cluster_bootstrap_resamples": resamples,
                "cluster_bootstrap_seed": seed,
                "cluster_unit": "cleaned_statement",
            }
        ]
    )
    diagnostics = diagnostics.assign(analysis="cooccurrence OR", panel="full_corpus")
    return summary, replicates, diagnostics


def load_table_ii_panel(
    panel_root: Path,
    normalize_label,
) -> tuple[pd.Series, np.ndarray, dict[str, np.ndarray]]:
    """Load and validate the retained ML/DL test predictions for one panel."""
    ml_path = panel_root / "outputs/ml_models_predictions.csv"
    dl_path = panel_root / "outputs/dl_models_predictions.csv"
    ml_cols = [
        "statement",
        "true_label",
        "lr_pred",
        "svm_pred",
        "rf_pred",
        "lgbm_pred",
    ]
    dl_cols = [
        "text",
        "true_label",
        "gru_pred",
        "cnn_pred",
        "albert_pred",
        "biobert_pred",
    ]
    ml = pd.read_csv(ml_path, usecols=ml_cols)
    dl = pd.read_csv(dl_path, usecols=dl_cols)
    if len(ml) != len(dl):
        raise AssertionError(f"ML/DL prediction row counts differ in {panel_root.name}")
    ml_text = ml["statement"].fillna("").astype(str)
    dl_text = dl["text"].fillna("").astype(str)
    if not ml_text.equals(dl_text):
        raise AssertionError(f"ML/DL test texts differ in {panel_root.name}")
    ml_true = normalize_label(ml["true_label"])
    dl_true = normalize_label(dl["true_label"])
    if not ml_true.equals(dl_true):
        raise AssertionError(f"ML/DL true labels differ in {panel_root.name}")
    mapping = {label: index for index, label in enumerate(EXPECTED_LABELS)}
    if set(ml_true.unique()) != set(EXPECTED_LABELS):
        raise ValueError(f"Incomplete label set in {panel_root.name}")
    y_true = ml_true.map(mapping).to_numpy(dtype=int)
    predictions: dict[str, np.ndarray] = {}
    for model, column in (
        ("LR", "lr_pred"),
        ("SVM", "svm_pred"),
        ("RF", "rf_pred"),
        ("LGBM", "lgbm_pred"),
    ):
        predictions[model] = label_codes(ml[column], EXPECTED_LABELS, normalize_label)
    for model, column in (
        ("GRU", "gru_pred"),
        ("CNN", "cnn_pred"),
        ("ALBERT", "albert_pred"),
        ("BioBERT", "biobert_pred"),
    ):
        predictions[model] = label_codes(dl[column], EXPECTED_LABELS, normalize_label)
    return ml_text, y_true, predictions


def load_table_iii_panel(path: Path) -> tuple[pd.Series, np.ndarray, dict[str, np.ndarray]]:
    columns = ["statement", "y_hard", "albert_pred", "biobert_pred"]
    df = pd.read_csv(path, usecols=columns)
    y_true = pd.to_numeric(df["y_hard"], errors="raise").to_numpy(dtype=int)
    predictions = {
        "ALBERT": pd.to_numeric(df["albert_pred"], errors="raise").to_numpy(dtype=int),
        "BioBERT": pd.to_numeric(df["biobert_pred"], errors="raise").to_numpy(dtype=int),
    }
    if set(np.unique(y_true)) != set(range(7)):
        raise ValueError(f"Incomplete hard-label codes in {path}")
    return df["statement"].fillna("").astype(str), y_true, predictions


def load_table_iv_tasks(path: Path) -> tuple[pd.Series, dict[tuple[str, str], tuple[np.ndarray, np.ndarray]]]:
    aspects = (
        "depression",
        "anxiety",
        "suicidal",
        "stress",
        "bipolar",
        "personality_disorder",
    )
    columns = ["statement"]
    for aspect in aspects:
        columns.extend(
            [
                f"{aspect}_true",
                f"albert_{aspect}_pred",
                f"biobert_{aspect}_pred",
            ]
        )
    df = pd.read_csv(path, usecols=columns)
    tasks: dict[tuple[str, str], tuple[np.ndarray, np.ndarray]] = {}
    for aspect in aspects:
        truth = pd.to_numeric(df[f"{aspect}_true"], errors="raise").to_numpy(dtype=int)
        if set(np.unique(truth)) != {0, 1, 2}:
            raise ValueError(f"Incomplete three-class target for {aspect}")
        for model, prefix in (("ALBERT", "albert"), ("BioBERT", "biobert")):
            pred = pd.to_numeric(
                df[f"{prefix}_{aspect}_pred"], errors="raise"
            ).to_numpy(dtype=int)
            tasks[(model, aspect)] = (truth, pred)
    return df["statement"].fillna("").astype(str), tasks


def _write_csv(table: pd.DataFrame, path: Path) -> None:
    table.to_csv(path, index=False, lineterminator="\n")


def _hash_outputs(output_dir: Path, excluded: Iterable[str]) -> pd.DataFrame:
    excluded_names = set(excluded)
    rows = []
    for path in sorted(p for p in output_dir.iterdir() if p.is_file()):
        if path.name in excluded_names:
            continue
        rows.append(
            {"file": path.name, "bytes": path.stat().st_size, "sha256": sha256_file(path)}
        )
    return pd.DataFrame(rows)


def run_analysis(r2_root: Path, output_dir: Path) -> dict[str, object]:
    r2_root = r2_root.resolve()
    output_dir = output_dir.resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    canonical = load_canonical_entropy_module(r2_root)
    full_data = r2_root / "02_mini4oLabel/data/mental_health_unified_labels_final.csv"

    table_i_coef, table_i_models, diag_table_i = fit_table_i_cluster_robust(
        full_data, canonical
    )
    cooccur_summary, cooccur_replicates, diag_cooccur = cooccurrence_cluster_bootstrap(
        full_data, canonical
    )

    table_ii_summaries: list[pd.DataFrame] = []
    table_ii_replicates: list[pd.DataFrame] = []
    table_ii_diagnostics: list[pd.DataFrame] = []
    for folder, panel in (
        ("01_OriginalLabel", "A_original_labels"),
        ("02_mini4oLabel", "B_ai_hard_labels"),
        ("03_SameLabel", "C_agreement_subset"),
    ):
        text, truth, predictions = load_table_ii_panel(
            r2_root / folder, canonical.normalize_label
        )
        groups = text.map(canonical.clean_text)
        summary, replicates, diagnostics = bootstrap_common_target_f1(
            analysis="Table II",
            panel=panel,
            groups=groups,
            y_true=truth,
            predictions=predictions,
            n_classes=7,
        )
        table_ii_summaries.append(summary)
        table_ii_replicates.append(replicates)
        table_ii_diagnostics.append(diagnostics)
    table_ii_summary = pd.concat(table_ii_summaries, ignore_index=True)
    table_ii_reps = pd.concat(table_ii_replicates, ignore_index=True)

    table_iii_summaries: list[pd.DataFrame] = []
    table_iii_replicates: list[pd.DataFrame] = []
    table_iii_diagnostics: list[pd.DataFrame] = []
    for relative, panel in (
        (
            "04_SoftLabel_TrainOnly/outputs/test_predictions_soft_train.csv",
            "A_soft_train_hard_validation_test",
        ),
        (
            "05_SoftLabel_TrainAll/outputs/test_predictions_soft_all.csv",
            "B1_soft_train_validation_test_hard_metrics",
        ),
    ):
        text, truth, predictions = load_table_iii_panel(r2_root / relative)
        groups = text.map(canonical.clean_text)
        summary, replicates, diagnostics = bootstrap_common_target_f1(
            analysis="Table III",
            panel=panel,
            groups=groups,
            y_true=truth,
            predictions=predictions,
            n_classes=7,
        )
        table_iii_summaries.append(summary)
        table_iii_replicates.append(replicates)
        table_iii_diagnostics.append(diagnostics)
    table_iii_summary = pd.concat(table_iii_summaries, ignore_index=True)
    table_iii_reps = pd.concat(table_iii_replicates, ignore_index=True)

    table_iv_text, table_iv_tasks = load_table_iv_tasks(
        r2_root / "06_AspectLabel/outputs/test_predictions_aspect.csv"
    )
    table_iv_summary, table_iv_reps, diag_table_iv = bootstrap_aspect_f1(
        groups=table_iv_text.map(canonical.clean_text),
        tasks=table_iv_tasks,
    )

    diagnostics = pd.concat(
        [
            diag_table_i,
            diag_cooccur,
            *table_ii_diagnostics,
            *table_iii_diagnostics,
            diag_table_iv,
        ],
        ignore_index=True,
    )
    status = pd.DataFrame(
        [
            {
                "analysis": "Table I logistic coefficients",
                "status": "completed",
                "method": "cleaned-statement cluster-robust sandwich covariance",
                "scope": "same maximum-likelihood point estimates; robust SE, p values, and CI",
            },
            {
                "analysis": "Co-occurrence odds ratio",
                "status": "completed",
                "method": "cleaned-statement cluster bootstrap",
                "scope": "2,000 percentile-bootstrap resamples of full text clusters",
            },
            {
                "analysis": "Table II weighted and macro F1 intervals",
                "status": "completed",
                "method": "cleaned-statement group bootstrap",
                "scope": "all eight retained models in Panels A, B, and C",
            },
            {
                "analysis": "Table III weighted and macro F1 intervals",
                "status": "completed",
                "method": "cleaned-statement group bootstrap",
                "scope": "ALBERT and BioBERT hard metrics in Panels A and B-1",
            },
            {
                "analysis": "Table IV weighted and macro F1 intervals",
                "status": "completed",
                "method": "cleaned-statement group bootstrap with shared post resamples",
                "scope": "all six heads and mean rows for ALBERT and BioBERT",
            },
            {
                "analysis": "Training-run variability",
                "status": "not_estimable_from_retained_predictions",
                "method": "none",
                "scope": "requires independent model refits; current sensitivity conditions on fitted seed-2025 models",
            },
        ]
    )

    tables = {
        "table_i_cluster_robust_coefficients.csv": table_i_coef,
        "table_i_cluster_robust_models.csv": table_i_models,
        "cooccurrence_cluster_bootstrap_summary.csv": cooccur_summary,
        "cooccurrence_cluster_bootstrap_replicates.csv": cooccur_replicates,
        "table_ii_group_bootstrap_f1_summary.csv": table_ii_summary,
        "table_ii_group_bootstrap_f1_replicates.csv": table_ii_reps,
        "table_iii_group_bootstrap_f1_summary.csv": table_iii_summary,
        "table_iii_group_bootstrap_f1_replicates.csv": table_iii_reps,
        "table_iv_group_bootstrap_f1_summary.csv": table_iv_summary,
        "table_iv_group_bootstrap_f1_replicates.csv": table_iv_reps,
        "cluster_diagnostics.csv": diagnostics,
        "analysis_completion_status.csv": status,
    }
    for filename, table in tables.items():
        _write_csv(table, output_dir / filename)

    input_paths = [
        full_data,
        r2_root / "01_OriginalLabel/outputs/ml_models_predictions.csv",
        r2_root / "01_OriginalLabel/outputs/dl_models_predictions.csv",
        r2_root / "02_mini4oLabel/outputs/ml_models_predictions.csv",
        r2_root / "02_mini4oLabel/outputs/dl_models_predictions.csv",
        r2_root / "03_SameLabel/outputs/ml_models_predictions.csv",
        r2_root / "03_SameLabel/outputs/dl_models_predictions.csv",
        r2_root / "04_SoftLabel_TrainOnly/outputs/test_predictions_soft_train.csv",
        r2_root / "05_SoftLabel_TrainAll/outputs/test_predictions_soft_all.csv",
        r2_root / "06_AspectLabel/outputs/test_predictions_aspect.csv",
    ]
    input_hashes = pd.DataFrame(
        [
            {
                "file": str(path.relative_to(r2_root)) if path.is_relative_to(r2_root) else str(path),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in input_paths
        ]
    )
    _write_csv(input_hashes, output_dir / "input_sha256.csv")

    excluded = {"output_sha256.csv", "cluster_sensitivity_manifest.json"}
    output_hashes = _hash_outputs(output_dir, excluded)
    _write_csv(output_hashes, output_dir / "output_sha256.csv")

    table_i_entropy = table_i_coef.loc[
        (table_i_coef["model"] == "entropy_only")
        & (table_i_coef["term"] == "entropy_std")
    ].iloc[0]
    adjusted_entropy = table_i_coef.loc[
        (table_i_coef["model"] == "entropy_plus_released_status")
        & (table_i_coef["term"] == "entropy_std")
    ].iloc[0]
    cooccur = cooccur_summary.iloc[0]
    manifest: dict[str, object] = {
        "analysis_id": "mental_health_r2_cluster_aware_sensitivity",
        "analysis_script": NOTEBOOK_NAME,
        "analysis_script_sha256": "n/a (code inlined in notebook)",
        "cluster_definition": (
            "lowercase; URL/user/hashtag token normalization; punctuation removal; "
            "whitespace collapse, via canonical entropy clean_text()"
        ),
        "full_corpus": {
            "n_rows": EXPECTED_N,
            "n_cleaned_statement_clusters": int(
                diag_table_i.iloc[0]["n_cleaned_statement_clusters"]
            ),
        },
        "table_i": {
            "entropy_only_or": float(table_i_entropy["odds_ratio"]),
            "entropy_only_cluster_robust_ci": [
                float(table_i_entropy["cluster_robust_or_ci_95_lower"]),
                float(table_i_entropy["cluster_robust_or_ci_95_upper"]),
            ],
            "adjusted_entropy_or": float(adjusted_entropy["odds_ratio"]),
            "adjusted_entropy_cluster_robust_ci": [
                float(adjusted_entropy["cluster_robust_or_ci_95_lower"]),
                float(adjusted_entropy["cluster_robust_or_ci_95_upper"]),
            ],
        },
        "cooccurrence": {
            "row_level_odds_ratio": float(cooccur["row_level_odds_ratio"]),
            "cluster_bootstrap_ci": [
                float(cooccur["cluster_bootstrap_ci_95_lower"]),
                float(cooccur["cluster_bootstrap_ci_95_upper"]),
            ],
            "bootstrap_resamples": OR_BOOTSTRAP_RESAMPLES,
            "bootstrap_seed": BOOTSTRAP_SEED,
        },
        "f1_intervals": {
            "bootstrap_resamples": F1_BOOTSTRAP_RESAMPLES,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "resampling_unit": "cleaned_statement",
            "tables_completed": ["Table II", "Table III hard metrics", "Table IV"],
            "conditioning": "retained fitted seed-2025 predictions",
            "does_not_include": "training-run variability",
        },
        "software": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
            "statsmodels": statsmodels.__version__,
            "platform": platform.platform(),
        },
        "input_hashes": "input_sha256.csv",
        "output_hashes": "output_sha256.csv",
        "output_file_count_excluding_manifest_and_hash_index": int(len(output_hashes)),
    }
    (output_dir / "cluster_sensitivity_manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    return manifest


In [3]:
manifest = run_analysis(R2_ROOT, OUTPUT)
print(json.dumps(manifest, indent=2, sort_keys=True))

{
  "analysis_id": "mental_health_r2_cluster_aware_sensitivity",
  "analysis_script": "MentalHealth_R2_cluster_sensitivity.ipynb",
  "analysis_script_sha256": "n/a (code inlined in notebook)",
  "cluster_definition": "lowercase; URL/user/hashtag token normalization; punctuation removal; whitespace collapse, via canonical entropy clean_text()",
  "cooccurrence": {
    "bootstrap_resamples": 2000,
    "bootstrap_seed": 2025,
    "cluster_bootstrap_ci": [
      1.4069745872549122,
      1.606793580776053
    ],
    "row_level_odds_ratio": 1.513541811041811
  },
  "f1_intervals": {
    "bootstrap_resamples": 1000,
    "bootstrap_seed": 2025,
    "conditioning": "retained fitted seed-2025 predictions",
    "does_not_include": "training-run variability",
    "resampling_unit": "cleaned_statement",
    "tables_completed": [
      "Table II",
      "Table III hard metrics",
      "Table IV"
    ]
  },
  "full_corpus": {
    "n_cleaned_statement_clusters": 50969,
    "n_rows": 53043
  },
  "inp

In [4]:
# Display the retained sensitivity tables backing Supplementary Table S4.
import pandas as pd

coef = pd.read_csv(OUTPUT / "table_i_cluster_robust_coefficients.csv")
coef_cols = ["model", "term", "odds_ratio",
             "model_based_or_ci_95_lower", "model_based_or_ci_95_upper",
             "cluster_robust_or_ci_95_lower", "cluster_robust_or_ci_95_upper",
             "cluster_robust_p_value"]
with pd.option_context("display.max_rows", None, "display.width", 220, "display.max_colwidth", 90):
    print("Cluster-robust logistic coefficients (Table I sensitivity):")
    print(coef[coef_cols].round(4).to_string(index=False))

    cooccur = pd.read_csv(OUTPUT / "cooccurrence_cluster_bootstrap_summary.csv")
    print("\nCo-occurrence odds ratio, cluster bootstrap:")
    print(cooccur.round(4).to_string(index=False))

    for name in ["table_ii_group_bootstrap_f1_summary.csv",
                 "table_iii_group_bootstrap_f1_summary.csv",
                 "table_iv_group_bootstrap_f1_summary.csv"]:
        frame = pd.read_csv(OUTPUT / name)
        print(f"\n{name}:")
        print(frame.round(4).to_string(index=False))

Cluster-robust logistic coefficients (Table I sensitivity):
                       model                                                                     term  odds_ratio  model_based_or_ci_95_lower  model_based_or_ci_95_upper  cluster_robust_or_ci_95_lower  cluster_robust_or_ci_95_upper  cluster_robust_p_value
                entropy_only                                                                Intercept      0.5898                      0.5794                      0.6004                         0.5646                         0.6162                  0.0000
                entropy_only                                                              entropy_std      1.3108                      1.2881                      1.3340                         1.2642                         1.3592                  0.0000
entropy_plus_released_status                                                                Intercept      0.1478                      0.1411                      0.1549   